# Refined datasets v2 · cajas YOLO reproducibles

Este notebook crea `v2` para LCW y Dogs of the World desde sus releases públicas e inmutables `v1`.

`v1` ya fue filtrado para conservar una mascota legible de la especie esperada, pero guarda la fotografía completa. `v2` no recorta ni duplica imágenes: agrega una caja YOLO verificada por foto para que el entrenamiento reproduzca el recorte de E2. Identidades, splits y bytes de imagen permanecen iguales.

| Entrada | Commit inmutable | Salida |
|---|---|---|
| `labeled-cats-in-the-wild-refined@v1` | `089518b5cffcf71a7f55dbc7d47d0f2269f98a2b` | `@v2` |
| `dogs-of-the-world-refined@v1` | `26a361971d5a77976fadd4aeae5e8294d40333c7` | `@v2` |


### Entorno

Usa Colab UI con GPU T4. La instalación viene de PyPI; `HF_TOKEN`, `MLFLOW_TRACKING_USERNAME` y `MLFLOW_TRACKING_PASSWORD` se leen de Colab Secrets. Drive conserva únicamente checkpoints de inferencia para reanudar una sesión interrumpida.


In [ ]:
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks]==0.2.21"

In [ ]:
import json, os, shutil, sys
from pathlib import Path

import torch
from google.colab import drive
import pettrace
from pettrace.data.workflow import configure_access
from pettrace.data.yolo_overlay import RELEASES

if sys.platform != "linux" or not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use Colab Linux with Python 3.11–3.13")
if pettrace.__version__ != "0.2.21":
    raise RuntimeError("Install PetTrace 0.2.21 before continuing")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; a T4 is sufficient")
os.environ["MLFLOW_TRACKING_URI"] = "https://ml.pettrace.app"
configure_access(names=("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"))
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
WORK = Path("/content/pettrace-yolo-v2")
CHECKPOINTS = Path("/content/drive/MyDrive/PetTrace/checkpoints/yolo-v2")
WORK.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage("/content").free < 60 * 2**30:
    raise OSError("At least 60 GiB of local scratch space is required")
print("Package:", pettrace.__version__, "| GPU:", torch.cuda.get_device_name(0))

## Fase 1. Verificar entradas y detector

Se resuelven los tags `v1` a sus commits auditados y se descarga únicamente el detector de E2 desde MLflow. Cualquier cambio de commit, manifiesto, shard o SHA-256 detiene el proceso.


In [ ]:
from pettrace.data.preparation import BASE_MODEL, DETECTOR_SHA256, acquire_detector
from pettrace.data.verification import sha256

detector_dir = acquire_detector(BASE_MODEL, WORK / "detector-models")
if sha256(detector_dir / "detector.onnx") != DETECTOR_SHA256:
    raise ValueError("The E2 detector hash changed")
for source, spec in RELEASES.items():
    print(source, "→", spec["repo_id"], "@", spec["commit"])
print("Detector:", BASE_MODEL, "|", DETECTOR_SHA256)

## Fase 2. Localización YOLO y evidencia v2

Cada TAR de v1 se verifica antes de extraerlo. CPU y GPU deben coincidir en la muestra de equivalencia antes de iniciar el barrido. La caché se respalda en Drive y permite reanudar sin recalcular detecciones confirmadas.

La salida local contiene solo `screening.csv.gz`, `gpu-validation.json` y `dataset.json`; las fotos siguen viviendo una sola vez en `versions/v1/data`.


In [ ]:
from pettrace.data.yolo_overlay import prepare_yolo_overlay

outputs = {}
for source in ("lcw", "dogs-world"):
    print(f"\n=== {source} ===", flush=True)
    outputs[source] = prepare_yolo_overlay(
        source, detector_dir, WORK, CHECKPOINTS, token=os.environ["HF_TOKEN"],
    )
    print("v2 local verificada:", outputs[source])

In [ ]:
from IPython.display import Markdown, display

rows = []
for source, output in outputs.items():
    manifest = json.loads((output / "dataset.json").read_text())
    rows.append(
        f"| {source} | {manifest['photos']:,} | {manifest['identities']:,} | "
        f"`{manifest['base']['commit'][:8]}` | `{manifest['files']['versions/v2/screening.csv.gz']['sha256'][:12]}` |"
    )
display(Markdown("| Fuente | Fotos | Identidades | Base v1 | Overlay SHA-256 |\n"
                 "|---|---:|---:|---|---|\n" + "\n".join(rows)))

## Fase 3. Publicar y documentar

Esta es la única celda que modifica Hugging Face. Añade el overlay y la documentación a cada repositorio, verifica de nuevo los archivos remotos y recién entonces crea el tag inmutable `v2`. Repetirla es seguro: un `v2` existente debe coincidir exactamente.


In [ ]:
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from pettrace.data.yolo_overlay import update_dataset_card

api = HfApi(token=os.environ["HF_TOKEN"])
published = {}
for source, output in outputs.items():
    spec = RELEASES[source]
    manifest = json.loads((output / "dataset.json").read_text())
    original_card = Path(hf_hub_download(
        spec["repo_id"], "README.md", repo_type="dataset", revision=spec["commit"],
        token=os.environ["HF_TOKEN"],
    )).read_text()
    expected_card = update_dataset_card(original_card, manifest)
    tags = {ref.name: ref.target_commit for ref in api.list_repo_refs(spec["repo_id"], repo_type="dataset").tags}
    if "v2" in tags:
        remote = json.loads(Path(hf_hub_download(
            spec["repo_id"], "versions/v2/dataset.json", repo_type="dataset", revision=tags["v2"],
            token=os.environ["HF_TOKEN"],
        )).read_text())
        remote_card = Path(hf_hub_download(
            spec["repo_id"], "README.md", repo_type="dataset", revision=tags["v2"],
            token=os.environ["HF_TOKEN"],
        )).read_text()
        if remote != manifest or remote_card != expected_card:
            raise ValueError(f"{spec['repo_id']}@v2 already identifies different evidence")
        published[source] = tags["v2"]
        print(source, "already published at", tags["v2"])
        continue

    card = output / "README.md"
    card.write_text(expected_card)
    names = ("dataset.json", "screening.csv.gz", "gpu-validation.json")
    operations = [CommitOperationAdd(path_in_repo="README.md", path_or_fileobj=str(card))]
    operations += [CommitOperationAdd(path_in_repo=f"versions/v2/{name}", path_or_fileobj=str(output / name))
                   for name in names]
    commit = api.create_commit(
        repo_id=spec["repo_id"], repo_type="dataset", operations=operations,
        commit_message="Publish pinned YOLO localization v2",
    )
    remote_manifest = json.loads(Path(hf_hub_download(
        spec["repo_id"], "versions/v2/dataset.json", repo_type="dataset", revision=commit.oid,
        token=os.environ["HF_TOKEN"],
    )).read_text())
    if remote_manifest != manifest:
        raise ValueError("Remote v2 manifest differs; no tag was created")
    remote_card = hf_hub_download(spec["repo_id"], "README.md", repo_type="dataset",
                                  revision=commit.oid, token=os.environ["HF_TOKEN"])
    if sha256(remote_card) != sha256(card):
        raise ValueError("Remote dataset card differs; no tag was created")
    for name in names[1:]:
        remote_path = hf_hub_download(spec["repo_id"], f"versions/v2/{name}", repo_type="dataset",
                                      revision=commit.oid, token=os.environ["HF_TOKEN"])
        if sha256(remote_path) != manifest["files"][f"versions/v2/{name}"]["sha256"]:
            raise ValueError(f"Remote file differs: {name}; no tag was created")
    api.create_tag(spec["repo_id"], tag="v2", repo_type="dataset", revision=commit.oid)
    published[source] = commit.oid
    print(f"Published: https://huggingface.co/datasets/{spec['repo_id']}/tree/v2")
print("Completed:", published)